In [1]:
#0 Load Libraries and Configurations

import os
import wrds
import pandas as pd
import numpy as np

# ---------- User-configurable paths ----------
PATH_DATA_INTERMEDIATE = "/Users/nglei/Desktop/Academics/SMU/Modules/QF600 Asset Pricing/Project/Project Code/cz_data/intermediate"  # <-- change this
os.makedirs(PATH_DATA_INTERMEDIATE, exist_ok=True)

OUT_PARQUET = os.path.join(PATH_DATA_INTERMEDIATE, "monthlyShortInterest.parquet")
OUT_CSV     = os.path.join(PATH_DATA_INTERMEDIATE, "monthlyShortInterest.csv")

In [ ]:
#1 Load Compustat Data

SQL = """
SELECT
    a.gvkey,
    a.iid,
    a.shortint,
    a.shortintadj,
    a.datadate
FROM comp.sec_shortint AS a
WHERE a.datadate >= DATE '2000-01-01'     -- ensure year 2000+
;
"""

In [ ]:
#2 Compustat Data Extraction From WRDS

db = wrds.Connection()  # prompts for creds if needed
df = db.raw_sql(SQL, date_cols=["datadate"])

In [ ]:
#3 Data Cleaning

# ------------- time_avail_m = mofd(datadate) -------------
# Stata %tm is a monthly index; we’ll store as month start timestamps.
df["time_avail_m"] = df["datadate"].dt.to_period("M").dt.to_timestamp("MS")

# ------------- Collapse monthly: first non-missing within month -------------
# Match gcollapse (firstnm) by sorting then taking first non-null in each group.
df = df.sort_values(["gvkey", "time_avail_m", "datadate"], kind="mergesort")

def first_non_missing(s: pd.Series):
    s = s.dropna()
    return s.iloc[0] if len(s) else np.nan

monthly = (
    df.groupby(["gvkey", "time_avail_m"], as_index=False)
      .agg({
          "shortint": first_non_missing,
          "shortintadj": first_non_missing
      })
)

# Optional: destring gvkey (numeric if possible)
monthly["gvkey"] = pd.to_numeric(monthly["gvkey"], errors="ignore")

# ------------- Save -------------
monthly.to_parquet(OUT_PARQUET, index=False)
monthly.to_csv(OUT_CSV, index=False)

print("Saved:")
print(" -", OUT_PARQUET)
print(" -", OUT_CSV)
print(monthly.head())